In [1]:
!pip install git+https://github.com/greydanus/mnist1d

  Cloning https://github.com/greydanus/mnist1d to /tmp/pip-req-build-b2o61an1
  Running command git clone --filter=blob:none --quiet https://github.com/greydanus/mnist1d /tmp/pip-req-build-b2o61an1
  Resolved https://github.com/greydanus/mnist1d to commit 7878d96082abd200c546a07a4101fa90b30fdf7e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mnist1d: filename=mnist1d-0.0.2.post16-py3-none-any.whl size=14665 sha256=69526023477f087dd85e313d49b4012d2d62197fe999d1bdd893effc6fcc3ffd
  Stored in directory: /tmp/pip-ephem-wheel-cache-3ex9kgih/wheels/33/99/c5/794fea38f039adfbca82fd56e4a00b3e1e4533a808baee9910
Successfully built mnist1d


In [2]:
import numpy as np
import os
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt
import mnist1d
import random

In [3]:
args = mnist1d.data.get_dataset_args()
data = mnist1d.data.get_dataset(args, path='./mnist1d_data.pkl', download=False, regenerate=False)

# The training and test input and outputs are in
# data['x'], data['y'], data['x_test'], and data['y_test']
print("Examples in training set: {}".format(len(data['y'])))
print("Examples in test set: {}".format(len(data['y_test'])))
print("Length of each example: {}".format(data['x'].shape[-1]))

Did or could not load data from ./mnist1d_data.pkl. Rebuilding dataset...
Examples in training set: 4000
Examples in test set: 1000
Length of each example: 40


In [15]:
train_data_x = data['x']
train_data_y = data['y']
test_data_x = data['x_test']
teat_data_y = data['y_test']

In [7]:
def print_variance(name, data):
  np_data = data.detach().numpy()
  neuron_variance = np.mean(np.var(np_data, axis=0))
  print("%s variance=%f"%(name,neuron_variance))

In [13]:
# He initialization of weights
def weights_init(layer_in):
  if isinstance(layer_in, nn.Linear):
    nn.init.kaiming_uniform_(layer_in.weight)
    layer_in.bias.data.fill_(0.0)

其中所有的量都是純量（scalar）。接著我們用這些統計量把批次（batch）內的活化值標準化，使其平均為 0、變異數為 1：

$$
\begin{aligned}
m_h &= \frac{1}{|\mathcal{B}|}\sum_{i\in\mathcal{B}} h_i \\[6pt]
s_h &= \sqrt{\frac{1}{|\mathcal{B}|}\sum_{i\in\mathcal{B}} \left(h_i - m_h\right)^2}
\end{aligned}
\tag{11.7}
$$

$$
h_i \leftarrow \frac{h_i - m_h}{s_h + \epsilon}
\qquad \forall i \in \mathcal{B},
\tag{11.8}
$$

其中 $\epsilon$ 是一個很小的數，用來避免除以零——當批次中所有 $h_i$ 都相同時 $s_h = 0$。

最後，標準化後的變數再乘上縮放參數 $\gamma$、加上平移參數 $\delta$：

$$
h_i \leftarrow \gamma h_i + \delta
\qquad \forall i \in \mathcal{B}.
\tag{11.9}
$$

> 參考：Appendix C.2.4 Standardization

In [25]:
def BatchNormalize(data, e, gamma, delta):
    B, H = data.shape
    out = np.zeros_like(data, dtype=float)

    for h in range(H):
        # 式 (11.7) 上半:m_h
        total = 0.0
        for i in range(B):
            total += data[i][h]
        mh = total / B

        # 式 (11.7) 下半:s_h
        total_square = 0.0
        for i in range(B):
            total_square += (data[i][h] - mh) ** 2
        sh = np.sqrt(total_square / B)

        # 式 (11.8) + (11.9)
        for i in range(B):
            out[i][h] = gamma[h] * (data[i][h] - mh) / (sh + e) + delta[h]

    return out

In [27]:
class ResidualNetwork(torch.nn.Module):
  def __init__(self,input_size,output_size,hidden_layer_size):
    super(ResidualNetwork, self).__init__()
    self.linear1 = nn.Linear(input_size,hidden_layer_size)
    self.linear2 = nn.Linear(hidden_layer_size,hidden_layer_size)
    self.linear3 = nn.Linear(hidden_layer_size,hidden_layer_size)
    self.linear4 = nn.Linear(hidden_layer_size,hidden_layer_size)
    self.linear5 = nn.Linear(hidden_layer_size,hidden_layer_size)
    self.linear6 = nn.Linear(hidden_layer_size,hidden_layer_size)
    self.linear7 = nn.Linear(hidden_layer_size,output_size)

  def forward(self,x):
    print_variance("Input",x)
    f = self.linear1(x)
    print_variance("First preactivation",f)
    res1 = f + self.linear2(f.relu())
    print_variance("First residual",res1)
    res2 = res1 + self.linear3(res1.relu())
    print_variance("Second residual",res2)
    res3 = res2 + self.linear4(res2.relu())
    print_variance("Third residual",res3)
    res4 = res3 + self.linear5(res3.relu())
    print_variance("Fourth residual",res4)
    res5 = res4 + self.linear6(res4.relu())
    print_variance("Fifth residual",res5)
    output = self.linear7(res5.relu())
    print_variance("Output",output)
    return output

In [33]:
class ResidualNetwork_BN(torch.nn.Module):
  def __init__(self,input_size,output_size,hidden_layer_size):
    super().__init__()
    self.linear1 = nn.Linear(input_size,hidden_layer_size)
    self.linear2 = nn.Linear(hidden_layer_size,hidden_layer_size)
    self.linear3 = nn.Linear(hidden_layer_size,hidden_layer_size)
    self.linear4 = nn.Linear(hidden_layer_size,hidden_layer_size)
    self.linear5 = nn.Linear(hidden_layer_size,hidden_layer_size)
    self.linear6 = nn.Linear(hidden_layer_size,hidden_layer_size)
    self.linear7 = nn.Linear(hidden_layer_size,output_size)

    self.bn1 = nn.BatchNorm1d(hidden_layer_size)
    self.bn2 = nn.BatchNorm1d(hidden_layer_size)
    self.bn3 = nn.BatchNorm1d(hidden_layer_size)
    self.bn4 = nn.BatchNorm1d(hidden_layer_size)
    self.bn5 = nn.BatchNorm1d(hidden_layer_size)

  def forward(self,x):
    print_variance("Input", x)
    f = self.linear1(x)
    print_variance("First preactivation", f)
    res1 = f    + self.linear2(self.bn1(f).relu())
    print_variance("First residual", res1)
    res2 = res1 + self.linear3(self.bn2(res1).relu())
    print_variance("Second residual", res2)
    res3 = res2 + self.linear4(self.bn3(res2).relu())
    print_variance("Third residual", res3)
    res4 = res3 + self.linear5(self.bn4(res3).relu())
    print_variance("Fourth residual", res4)
    res5 = res4 + self.linear6(self.bn5(res4).relu())
    print_variance("Fifth residual", res5)
    output = self.linear7(res5.relu())
    print_variance("Output", output)
    return output

In [31]:
n_hidden = 100
n_input = 40
n_output = 10

In [16]:
x_train = torch.tensor(train_data_x.astype('float32'))
y_train = torch.tensor(train_data_y.astype('long'))

In [21]:
def train_with_set_epoch(model,epochs,batch):
  loss_function = nn.CrossEntropyLoss()
  optimizer = torch.optim.SGD(model.parameters(), lr = 0.05, momentum=0.9)
  data_loader = DataLoader(TensorDataset(x_train,y_train), batch_size=batch, shuffle=True, worker_init_fn=np.random.seed(1))
  model.apply(weights_init)

  for epoch in range(epochs):
    for i,(x,y) in enumerate(data_loader):
      y_pred = model(x)
      loss = loss_function(y_pred,y)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

In [34]:
print("Residual Network without BN structure--------------")
Residualmodel = ResidualNetwork(n_input, n_output, n_hidden)
train_with_set_epoch(Residualmodel,1,100)
print("------------------------------------------------------")

print("Residual Network with BN structure")
# 建立類別實例 (Instantiate the model)
ResidualNetwork_BN_model = ResidualNetwork_BN(n_input, n_output, n_hidden)
train_with_set_epoch(ResidualNetwork_BN_model,1,100)
print("------------------------------------------------------")

Residual Network without BN structure--------------
Input variance=1.029245
First preactivation variance=2.070406
First residual variance=3.650501
Second residual variance=6.054786
Third residual variance=9.548914
Fourth residual variance=15.639400
Fifth residual variance=26.260597
Output variance=19.533899
Input variance=0.959661
First preactivation variance=1.878977
First residual variance=3.152964
Second residual variance=5.054112
Third residual variance=7.683903
Fourth residual variance=11.660583
Fifth residual variance=18.153008
Output variance=14.286244
Input variance=0.973029
First preactivation variance=1.904205
First residual variance=3.161593
Second residual variance=4.802560
Third residual variance=6.786934
Fourth residual variance=9.166994
Fifth residual variance=12.384691
Output variance=5.068334
Input variance=0.978748
First preactivation variance=1.904389
First residual variance=3.189148
Second residual variance=4.762908
Third residual variance=6.611446
Fourth residual v